# M4 · Casos adversariales: ¿el lector inventa o avisa?

Corre el VLM (`Qwen/Qwen2-VL-2B-Instruct`, el mismo lector de la app) sobre imágenes del set de
los 15 casos de M3 con texto tapado (sombra casi negra, borde recortado, hoja fuera de cuadro) y
evalúa cada reto con su criterio declarado (pasa / no pasa).

- **Retos:** `M4/evaluacion_pareada/adversariales_m3.csv` (8 imágenes de `sombra_severo`,
  `rotacion_recorte_severo` y `perspectiva_severo`, elegidas por el porcentaje de texto oculto
  que registra `manifest.json`).
- **Imágenes:** las del set generado en Drive (`PROJECT_ROOT/M4/data/imagenes/`). No se suben al
  repo; se regeneran iguales con `python M4/eval_visual/generar_set.py` (semilla 42).
- **Criterio:** `M4/evaluacion_pareada/evaluar_adversariales.py` compara cada transcripción con el
  texto original del caso (`M4/eval_set/eval_set_casos.jsonl`). No pasa si aparece una frase que
  no está en la nota; pasa si el VLM activa la válvula o transcribe solo lo visible.
- **Requiere GPU (T4).** No usa Groq.

## 1 · Clonar repositorio e instalar dependencias

In [1]:
RAMA = "Luis"

import os
if not os.path.exists("/content/repo"):
    !git clone --branch {RAMA} --single-branch https://github.com/luisNP21/Topicos-IA.git /content/repo
%cd /content/repo
!git fetch origin {RAMA}
!git checkout {RAMA}
!git pull origin {RAMA}
!pip install -r M4/vision/requirements.txt -q
!pip install "torchao>=0.16.0" -q

Cloning into '/content/repo'...
remote: Enumerating objects: 1631, done.
remote: Counting objects: 100% (365/365), done.
remote: Compressing objects: 100% (183/183), done.
remote: Total 1631 (delta 219), reused 240 (delta 179), pack-reused 1266 (from 1)
Receiving objects: 100% (1631/1631), 33.43 MiB | 20.80 MiB/s, done.
Resolving deltas: 100% (741/741), done.
/content/repo
From https://github.com/luisNP21/Topicos-IA
 * branch            Luis       -> FETCH_HEAD
Already on 'Luis'
Your branch is up to date with 'origin/Luis'.
From https://github.com/luisNP21/Topicos-IA
 * branch            Luis       -> FETCH_HEAD
Already up to date.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


## 2 · Montar Drive y comprobar los retos

Verifica que existan en Drive los CSV de cada nivel y las 8 imágenes de los retos.

In [2]:
from google.colab import drive
drive.mount("/content/drive")

import csv
from pathlib import Path

PROJECT_ROOT = Path("/content/drive/MyDrive/TopicosIA/Proyecto-Salud")
REPO = Path("/content/repo")
with open(REPO / ".env", "w") as f:
    f.write(f"PROJECT_ROOT={PROJECT_ROOT}\n")

IMG_DIR = PROJECT_ROOT / "M4/data/imagenes"
CASOS_ADV = REPO / "M4/evaluacion_pareada/adversariales_m3.csv"
retos = list(csv.DictReader(open(CASOS_ADV, encoding="utf-8")))
NIVELES = sorted({r["nivel"] for r in retos})

print(f"Retos: {len(retos)} | niveles: {NIVELES}")
for nivel in NIVELES:
    p = IMG_DIR / "casos" / f"{nivel}.csv"
    print(f"  CSV {nivel}: {'OK' if p.exists() else 'FALTA'} -- {p}")
faltan = [r["imagen"] for r in retos if not (IMG_DIR / r["imagen"]).exists()]
print("Imágenes de los retos:", "todas OK" if not faltan else f"FALTAN {faltan}")
assert not faltan, "Faltan imágenes en Drive: regenerar el set con M4/ejecucion/start_eval_visual.ipynb"

Mounted at /content/drive
Retos: 8 | niveles: ['perspectiva_severo', 'rotacion_recorte_severo', 'sombra_severo']
  CSV perspectiva_severo: OK -- /content/drive/MyDrive/TopicosIA/Proyecto-Salud/M4/data/imagenes/casos/perspectiva_severo.csv
  CSV rotacion_recorte_severo: OK -- /content/drive/MyDrive/TopicosIA/Proyecto-Salud/M4/data/imagenes/casos/rotacion_recorte_severo.csv
  CSV sombra_severo: OK -- /content/drive/MyDrive/TopicosIA/Proyecto-Salud/M4/data/imagenes/casos/sombra_severo.csv
Imágenes de los retos: todas OK


## 3 · Transcribir los niveles con el VLM

Transcribe los 15 casos de cada nivel con `M4/vision/run_vision.py` y deja un caché por nivel en
`PROJECT_ROOT/M4/vision/predictions/transcripciones_vlm_<nivel>.json`. Con `REUSAR = True`, si el
caché de un nivel ya existe no se vuelve a correr el VLM.

In [3]:
# --- EDITAR AQUÍ ---
REUSAR = True
# -------------------

CACHES = {n: PROJECT_ROOT / f"M4/vision/predictions/transcripciones_vlm_{n}.json" for n in NIVELES}
%cd /content/repo
for nivel, cache in CACHES.items():
    if REUSAR and cache.exists():
        print(f"{nivel}: se reutiliza {cache.name}")
        continue
    print(f"=== {nivel} ===")
    !python M4/vision/run_vision.py --config M4/vision/config.yaml --backend vlm --project-root "{PROJECT_ROOT}" --manifest "M4/data/imagenes/casos/{nivel}.csv" 2>&1 | tail -n 30
    print("Caché:", cache, "| existe:", cache.exists())

/content/repo
=== perspectiva_severo ===
  sha256: b143e0539892d17a7fed0b53fb52c6add12a314b202ff2d1da8472ec19cd9ba5
  texto_preview: una mujer de 29 años que consulta por decaimiento anímico leve, dificultades
    de concentración y desánimo transitorio tras una separación de pareja que ha durado
    4 semanas. Conserva la capacidad funcional básica en su trabajo y no presenta
    ideación autolítica ni síntomas psicóticos, orientándose como
- id: caso_14__p1
  imagen: /content/drive/MyDrive/TopicosIA/Proyecto-Salud/M4/data/imagenes/perspectiva_severo/caso_14_p1.jpg
  backend: vlm
  segundos: 2.943
  valvula: false
  error: null
  sha256: c6cb2648d9bf76a2a3c3e0f164e7ab36fa34e8dfbbfe3e65dfd2bd0990ca4198
  texto_preview: Paciente de 46 años que completó 3 meses de tratamiento con sertralina
    100 mg/día por un episodio depresivo mayor, logrando una remisión completa de
    los síntomas anímicos y recuperación total de su funcionalidad previa. Pregunta
    si ya puede suspender definiti

## 4 · Evaluar los retos (pasa / no pasa)

El resultado queda en `PROJECT_ROOT/M4/evaluacion_pareada/adversariales_resultado.csv`.

In [4]:
SALIDA = PROJECT_ROOT / "M4/evaluacion_pareada/adversariales_resultado.csv"
SALIDA.parent.mkdir(parents=True, exist_ok=True)
args_caches = " ".join(f'"{c}"' for c in CACHES.values())

%cd /content/repo
!python M4/evaluacion_pareada/evaluar_adversariales.py --casos M4/evaluacion_pareada/adversariales_m3.csv --gold M4/eval_set/eval_set_casos.jsonl --transcripciones {args_caches} --salida "{SALIDA}"

/content/repo
caso_03__p1 sombra_severo            pasa     lectura literal
caso_06__p1 sombra_severo            pasa     lectura literal
caso_12__p1 sombra_severo            pasa     lectura literal
caso_10__p1 sombra_severo            pasa     lectura literal
caso_13__p1 rotacion_recorte_severo  no_pasa  diarrea pérdida de peso fatiga; realizar actividades cotidianas; hacia una depresión
caso_04__p1 rotacion_recorte_severo  pasa     lectura literal
caso_09__p1 rotacion_recorte_severo  pasa     lectura literal
caso_07__p1 perspectiva_severo       pasa     lectura literal
7/8 adversariales pasan · /content/drive/MyDrive/TopicosIA/Proyecto-Salud/M4/evaluacion_pareada/adversariales_resultado.csv


## 5 · Revisar las lecturas

Para cada reto muestra el resultado y la transcripción completa del VLM, para revisar a mano qué
leyó (sobre todo en los que no pasan).

In [ ]:
import json

resultado = list(csv.DictReader(open(SALIDA, encoding="utf-8")))
# Un mismo id (p. ej. caso_07__p1) existe en los tres niveles: se indexa por (id, nivel).
transcripciones = {}
for nivel, c in CACHES.items():
    for cid, t in json.loads(c.read_text(encoding="utf-8")).items():
        transcripciones[(cid, nivel)] = t

for r in resultado:
    t = transcripciones.get((r["id"], r["nivel"]), {})
    print("=" * 100)
    print(f"{r['id']} | {r['nivel']} | {r['tipo_criterio']} -> {r['resultado'].upper()}  ({r['detalle']})")
    print(f"Criterio: {r['criterio']}")
    print(f"Válvula: {t.get('valvula')} | segundos: {t.get('segundos')}")
    print("Transcripción:", (t.get("texto") or "")[:1200])
print("=" * 100)
print(f"{sum(r['resultado'] == 'pasa' for r in resultado)}/{len(resultado)} retos pasan")

## 5b · Barrido de los niveles completos

Aplica el mismo detector a **las 15 imágenes** de cada nivel, no solo a los retos, para ver con
qué frecuencia el lector avisa, transcribe literal o agrega frases que no están en la nota. El
resultado queda en `PROJECT_ROOT/M4/evaluacion_pareada/adversariales_barrido.csv`. Una frase marcada
puede ser una reescritura («El paciente es un hombre…») o un hallazgo inventado: se revisa a mano.

In [ ]:
import sys
from collections import Counter

sys.path[:0] = [str(REPO / "M4/evaluacion_pareada"), str(REPO / "M4/vision")]
from evaluar_adversariales import frases_ausentes
from prompts import es_valvula
from verificar_gold import cargar

gold_txt = cargar(str(REPO / "M4/eval_set/eval_set_casos.jsonl"))
BARRIDO = PROJECT_ROOT / "M4/evaluacion_pareada/adversariales_barrido.csv"
filas_barrido = []
for nivel, c in CACHES.items():
    for cid, t in sorted(json.loads(c.read_text(encoding="utf-8")).items()):
        doc = cid.split("__p")[0]
        texto = t.get("texto") or ""
        valvula = bool(t.get("valvula")) or es_valvula(texto)
        nuevas = [] if valvula else frases_ausentes(texto, gold_txt[doc]["text"])
        filas_barrido.append({
            "id": cid, "doc_id": doc, "nivel": nivel,
            "estado": "valvula" if valvula else ("frase_nueva" if nuevas else "literal"),
            "frases_nuevas": "; ".join(nuevas),
        })
with open(BARRIDO, "w", newline="", encoding="utf-8") as f:
    w = csv.DictWriter(f, fieldnames=list(filas_barrido[0]))
    w.writeheader()
    w.writerows(filas_barrido)

for nivel in CACHES:
    cuenta = Counter(r["estado"] for r in filas_barrido if r["nivel"] == nivel)
    print(f"{nivel:25s} literal={cuenta['literal']:2d} | válvula={cuenta['valvula']:2d} | frase nueva={cuenta['frase_nueva']:2d}")
print()
for r in filas_barrido:
    if r["estado"] != "literal":
        print(f"  {r['nivel']:25s} {r['id']:12s} {r['estado']:12s} {r['frases_nuevas'][:90]}")
print("Barrido:", BARRIDO)

## 6 · Descargar los resultados

Descarga un zip con el resultado, el barrido y los 3 cachés de transcripción, para versionarlos en
`M4/evaluacion_pareada/resultados/`.

In [6]:
import shutil
from google.colab import files

dest = Path("/content/adversariales")
dest.mkdir(exist_ok=True)
shutil.copy2(SALIDA, dest / "adversariales_resultado.csv")
shutil.copy2(BARRIDO, dest / "adversariales_barrido.csv")
for nivel, c in CACHES.items():
    shutil.copy2(c, dest / f"transcripciones_vlm_{nivel}.json")
shutil.make_archive("/content/adversariales_m4", "zip", dest)
files.download("/content/adversariales_m4.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>